# Code complexity × proposal outcomes

Does the cyclomatic complexity of the codebase at the time a proposal is introduced
predict how quickly it will be accepted, or whether it will be accepted at all?

**Data sources:**
- `complexity.db` — semiannual snapshots of per-function Lizard and per-file SCC metrics.
- `all_proposals.db` — proposal metadata, status history, and revisions.

**Approach (per project, independently):**
1. Compute **mean** project-wide cyclomatic complexity per
   snapshot, plus **total SLOC** (sum of `code_lines` from `FileMetric`).
2. Interpolate these semiannual series to the exact date each proposal was first seen.
3. Formulate and test hypotheses with **Kendall's τ**.

**Hypotheses:**
- **H1** — Higher code complexity / larger codebase at proposal introduction is
  associated with *longer* time to acceptance.
- **H2** — Higher code complexity / larger codebase at proposal introduction is
  associated with a *lower* probability of acceptance.

Independent variables: Mean CC, Total SLOC.

In [ ]:
import sqlite3

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats, interpolate

# Code complexity × proposal outcomes

Does
the
cyclomatic
complexity
of
the
codebase
at
the
time
a
proposal is introduced
predict
how
quickly
it
will
be
accepted, or whether
it
will
be
accepted
at
all?

** Data
sources: **
- `complexity.db` — semiannual
snapshots
of
per - function
Lizard and per - file
SCC
metrics.
- `all_proposals.db` — proposal
metadata, status
history, and revisions.

** Approach(per
project, independently): **
1.
Compute ** mean ** project - wide
cyclomatic
complexity
per
snapshot, plus ** total
SLOC ** (sum of `code_lines` from `FileMetric`).
2.
Interpolate
these
semiannual
series
to
the
exact
date
each
proposal
was
first
seen.
3.
Formulate and test
hypotheses
with ** Kendall's τ**.

** Hypotheses: **
- ** H1 ** — Higher
code
complexity / larger
codebase
at
proposal
introduction is
associated
with *longer * time to acceptance.
- ** H2 ** — Higher
code
complexity / larger
codebase
at
proposal
introduction is
associated
with a * lower * probability of acceptance.

Independent
variables: Mean
CC, Total
SLOC.

In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats, interpolate

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

%config
InlineBackend.figure_format = 'retina'

In [ ]:
def _find_db(name):
    candidates = [Path(f"data/{name}"), Path(f"../data/{name}"),
                  Path(f"../../data/{name}"), Path(f"../../../data/{name}")]
    p = next((c.resolve() for c in candidates if c.exists()), None)
    if p is None:
        raise FileNotFoundError(f"{name} not found")
    print(f"Using {p}")
    return p

complexity_conn = sqlite3.connect(_find_db("complexity/complexity.db"))
proposals_conn = sqlite3.connect(_find_db("shared/all_proposals.db"))

## 1 · Discover projects & load complexity data

Map `project_id`s from `complexity.db` to project names
from `all_proposals.db`, and compute per-snapshot aggregates (complexity percentiles
and total SLOC) for each project independently.

In [ ]:
# Project name mapping from all_proposals.db
project_names = dict(pd.read_sql_query(
    "SELECT project_id, project_name FROM Project", proposals_conn
).values)

# Discover which project_ids have snapshots in complexity.db
available_ids = pd.read_sql_query(
    "SELECT DISTINCT project_id FROM Snapshot ORDER BY project_id", complexity_conn
)["project_id"].tolist()

print(f"Found projects: {[project_names.get(pid, f'?{pid}') for pid in available_ids]}")

sql_latest = """
             WITH latest AS (SELECT snapshot_id,
                                    ROW_NUMBER() OVER (
                                      PARTITION BY project_id, quarter_label
                                      ORDER BY analyzed_at DESC
                                      ) AS rn
                             FROM Snapshot) \
             """

# Load per-function complexity with project_id via Snapshot
func_cc = pd.read_sql_query(f"""
    {sql_latest}
    SELECT s.project_id, s.snapshot_id, s.quarter_label, s.commit_time,
           f.cyclomatic_complexity
    FROM Snapshot s
    JOIN latest l ON l.snapshot_id = s.snapshot_id AND l.rn = 1
    JOIN FunctionMetric f ON f.snapshot_id = s.snapshot_id
""", complexity_conn)

# Aggregate to per-snapshot cyclomatic stats
snapshot_complexity = (
    func_cc.groupby(["project_id", "snapshot_id", "quarter_label", "commit_time"])
    ["cyclomatic_complexity"]
    .agg(
        n_functions="count",
        cc_mean="mean",
    )
    .reset_index()
)
del func_cc

# Total SLOC per snapshot from FileMetric
file_sloc = pd.read_sql_query(f"""
    {sql_latest}
    SELECT s.project_id, s.snapshot_id, fm.file_path, fm.code_lines
    FROM Snapshot s
    JOIN latest l ON l.snapshot_id = s.snapshot_id AND l.rn = 1
    JOIN FileMetric fm ON fm.snapshot_id = s.snapshot_id
""", complexity_conn)

snapshot_sloc = (
    file_sloc.groupby("snapshot_id")["code_lines"]
    .sum()
    .reset_index(name="total_sloc")
)
del file_sloc

snapshot_complexity = snapshot_complexity.merge(snapshot_sloc, on="snapshot_id", how="left")
snapshot_complexity["total_sloc"] = snapshot_complexity["total_sloc"].fillna(0).astype(int)
snapshot_complexity["commit_time"] = pd.to_datetime(snapshot_complexity["commit_time"], utc=True)
snapshot_complexity = snapshot_complexity.sort_values(["project_id", "commit_time"])
snapshot_complexity["project_name"] = snapshot_complexity["project_id"].map(project_names)

for pid in available_ids:
    sub = snapshot_complexity[snapshot_complexity["project_id"] == pid]
    print(f"\n{project_names[pid]} (id={pid}): {len(sub)} snapshots, "
          f"{sub['commit_time'].min().strftime('%Y-%m')} – {sub['commit_time'].max().strftime('%Y-%m')}")

In [ ]:
# Per-project complexity & SLOC timeline
for pid in available_ids:
    pname = project_names[pid]
    sub = snapshot_complexity[snapshot_complexity["project_id"] == pid]

    fig, (ax1, ax3) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.plot(sub["commit_time"], sub["cc_mean"], "s--", label="Mean", color="green")
    ax1.set_ylabel("Cyclomatic complexity")
    ax1.set_title(f"{pname} — cyclomatic complexity over time")
    ax1.legend()
    ax2 = ax1.twinx()
    ax2.bar(sub["commit_time"], sub["n_functions"], width=60, alpha=0.15, color="gray", label="# functions")
    ax2.set_ylabel("# functions analysed")
    ax2.legend(loc="upper left")

    ax3.plot(sub["commit_time"], sub["total_sloc"] / 1e6, "D-", color="darkgreen")
    ax3.set_xlabel("Snapshot date")
    ax3.set_ylabel("Total SLOC (millions)")
    ax3.set_title(f"{pname} — total source lines of code over time")

    plt.tight_layout()
    plt.show()

## 2 · Proposal base table (all projects)

One row per proposal with `first_seen_at`, terminal status, and time to terminal
decision. Only projects that exist in `complexity.db` are kept.

In [ ]:
all_proposals = pd.read_sql_query("""
                                  WITH first_seen AS (SELECT project_id, proposal_id, MIN(created_at) AS first_seen_at
                                                      FROM (SELECT project_id, proposal_id, created_at
                                                            FROM ProposalRevision
                                                            UNION ALL
                                                            SELECT project_id, proposal_id, created_at
                                                            FROM ProposalStatus)
                                                      GROUP BY project_id, proposal_id),
                                       latest AS (SELECT project_id,
                                                         proposal_id,
                                                         normalised_status AS latest_status,
                                                         MAX(created_at)   AS latest_status_at
                                                  FROM ProposalStatus
                                                  GROUP BY project_id, proposal_id),
                                       first_terminal AS (SELECT project_id,
                                                                 proposal_id,
                                                                 MIN(created_at)   AS first_terminal_at,
                                                                 normalised_status AS terminal_status
                                                          FROM ProposalStatus
                                                          WHERE normalised_status IN ('accepted', 'rejected', 'withdrawn', 'superseded')
                                                          GROUP BY project_id, proposal_id)
                                  SELECT p.project_id,
                                         p.proposal_id,
                                         fs.first_seen_at,
                                         l.latest_status,
                                         ft.first_terminal_at,
                                         ft.terminal_status
                                  FROM Proposal p
                                         JOIN first_seen fs USING (project_id, proposal_id)
                                         JOIN latest l USING (project_id, proposal_id)
                                         LEFT JOIN first_terminal ft USING (project_id, proposal_id)
                                  """, proposals_conn)

_parse = lambda col: pd.to_datetime(col, errors="coerce", utc=True, format="mixed")
all_proposals["first_seen_at"] = _parse(all_proposals["first_seen_at"])
all_proposals["first_terminal_at"] = _parse(all_proposals["first_terminal_at"])

# Keep only projects that exist in complexity.db
all_proposals = all_proposals[all_proposals["project_id"].isin(available_ids)].copy()
all_proposals["project_name"] = all_proposals["project_id"].map(project_names)

for pid in available_ids:
    sub = all_proposals[all_proposals["project_id"] == pid]
    ts = sub["terminal_status"].value_counts(dropna=False)
    print(f"{project_names[pid]}: {len(sub)} proposals — {ts.to_dict()}")

## 3 · Interpolate complexity and SLOC to proposal dates (per project)

For each project independently, build linear interpolation of the quarterly
complexity / SLOC series and evaluate at each proposal's `first_seen_at`.
Proposals outside the snapshot time range are excluded.

In [ ]:
IV_COLS = ["cc_mean", "total_sloc"]
IV_LABELS = ["Mean CC", "Total SLOC"]

proposals_with_iv = []

for pid in available_ids:
    pname = project_names[pid]
    snap = snapshot_complexity[snapshot_complexity["project_id"] == pid].sort_values("commit_time")
    props = all_proposals[all_proposals["project_id"] == pid].copy()

    if len(snap) < 2:
        print(f"{pname}: <2 snapshots, skipping interpolation")
        continue

    snap_ts = snap["commit_time"].astype(np.int64) / 1e9
    snap_min, snap_max = snap["commit_time"].min(), snap["commit_time"].max()

    # Build interpolators for each IV
    interps = {}
    for col in IV_COLS:
        interps[col] = interpolate.interp1d(snap_ts, snap[col], kind="linear", bounds_error=True)

    # Filter proposals to snapshot range
    in_range = props[props["first_seen_at"].between(snap_min, snap_max)].copy()
    if in_range.empty:
        print(f"{pname}: no proposals within snapshot range, skipping")
        continue

    prop_ts = in_range["first_seen_at"].astype(np.int64) / 1e9
    for col in IV_COLS:
        in_range[col] = interps[col](prop_ts)

    proposals_with_iv.append(in_range)
    print(f"{pname}: {len(in_range)}/{len(props)} proposals within "
          f"{snap_min.strftime('%Y-%m')} – {snap_max.strftime('%Y-%m')}")

proposals_with_iv = pd.concat(proposals_with_iv, ignore_index=True)
print(f"\nTotal proposals with interpolated IVs: {len(proposals_with_iv)}")

## 4 · Derived variables

- **`days_to_terminal`** — calendar days from `first_seen_at` to `first_terminal_at`
  (only for proposals that reached a terminal status).
- **`accepted`** — binary indicator (1 = accepted, 0 = rejected), restricted to
  proposals that reached one of these two terminal outcomes. Withdrawn and superseded
  proposals are excluded from H2 because they do not represent a quality judgement.

In [ ]:
# H1 dataset: accepted proposals with days_to_acceptance > 0 (per project)
accepted = proposals_with_iv[proposals_with_iv["terminal_status"] == "accepted"].copy()
accepted["days_to_acceptance"] = (
        (accepted["first_terminal_at"] - accepted["first_seen_at"]).dt.total_seconds() / 86400
)
accepted = accepted[accepted["days_to_acceptance"] > 0]

# H2 dataset: accepted vs rejected (binary outcome, per project)
decided = proposals_with_iv[
    proposals_with_iv["terminal_status"].isin(["accepted", "rejected"])
].copy()
decided["accepted"] = (decided["terminal_status"] == "accepted").astype(int)

for pid in available_ids:
    pname = project_names[pid]
    h1 = accepted[accepted["project_id"] == pid]
    h2 = decided[decided["project_id"] == pid]
    n_acc = h2["accepted"].sum() if not h2.empty else 0
    n_rej = len(h2) - n_acc
    print(f"{pname}: H1 n={len(h1)} (accepted, >0 days)  |  "
          f"H2 n={len(h2)} ({n_acc} accepted, {n_rej} rejected)")

## 5 · Hypothesis testing — Kendall's τ

Kendall's rank correlation coefficient is used because:
- It makes no distributional assumptions (non-parametric).
- It is robust to ties and outliers.
- It handles ordinal / continuous variables equally well.

**Significance level:** α = 0.05 (two-tailed).

In [ ]:
ALPHA = 0.05
MIN_N = 5  # minimum sample size for a meaningful test


def kendall_test(x, y):
    """Run Kendall's tau, return (tau, p)."""
    tau, p = stats.kendalltau(x, y)
    return tau, p


all_results = []

for pid in available_ids:
    pname = project_names[pid]
    h1 = accepted[accepted["project_id"] == pid]
    h2 = decided[decided["project_id"] == pid]

    print("=" * 70)
    print(f"  {pname}")
    print("=" * 70)

    # H1: IV vs days to acceptance
    if len(h1) >= MIN_N:
        print(f"\n  H1: complexity/SLOC → days to acceptance  (n={len(h1)})\n")
        for suffix, col, label in zip("abcde", IV_COLS, IV_LABELS):
            x, y = h1[col], h1["days_to_acceptance"]
            if x.nunique() < 2:
                print(f"    H1{suffix} [{label}]: constant IV, skipped")
                continue
            tau, p = kendall_test(x, y)
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < ALPHA else "n.s."
            print(f"    H1{suffix} [{label}]:  τ = {tau:+.4f},  p = {p:.4e}  {sig}")
            all_results.append({"project": pname, "hypothesis": f"H1{suffix}",
                                "iv": label, "dv": "Days to acceptance",
                                "n": len(h1), "tau": tau, "p": p,
                                "significant": p < ALPHA})
    else:
        print(f"\n  H1: skipped (n={len(h1)} < {MIN_N})")

    # H2: IV vs accepted/rejected
    has_both = h2["accepted"].nunique() >= 2 if len(h2) >= MIN_N else False
    if has_both:
        n_acc = h2["accepted"].sum()
        n_rej = len(h2) - n_acc
        print(f"\n  H2: complexity/SLOC → acceptance likelihood  "
              f"(n={len(h2)}, {n_acc} acc / {n_rej} rej)\n")
        for suffix, col, label in zip("abcde", IV_COLS, IV_LABELS):
            x, y = h2[col], h2["accepted"]
            if x.nunique() < 2:
                print(f"    H2{suffix} [{label}]: constant IV, skipped")
                continue
            tau, p = kendall_test(x, y)
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < ALPHA else "n.s."
            print(f"    H2{suffix} [{label}]:  τ = {tau:+.4f},  p = {p:.4e}  {sig}")
            all_results.append({"project": pname, "hypothesis": f"H2{suffix}",
                                "iv": label, "dv": "Accepted (1) vs Rejected (0)",
                                "n": len(h2), "tau": tau, "p": p,
                                "significant": p < ALPHA})
    else:
        print(f"\n  H2: skipped (n={len(h2)}, need ≥{MIN_N} with both outcomes)")

    print()

results_df = pd.DataFrame(all_results)

## 7 · Summary table and interpretation

In [ ]:
summary = results_df.copy()
summary["tau"] = summary["tau"].map("{:+.4f}".format)
summary["p"] = summary["p"].map("{:.4e}".format)
summary["verdict"] = summary["significant"].map({True: "SUPPORTED", False: "NOT SUPPORTED"})
summary[["project", "hypothesis", "iv", "dv", "n", "tau", "p", "verdict"]]

## 8 . RQ2 result figures

In [ ]:
from pathlib import Path
PLOTS = Path("plots")
PLOTS.mkdir(exist_ok=True)

PAPER_IVS = ["Mean CC", "Total SLOC"]
PROJECT_ORDER = ["Python", "Pandas", "NumPy", "JavaScript", "C++", "Rust",
                 "OpenJDK", "Swift", "Kubernetes", "Kotlin"]

rq2 = results_df[results_df["iv"].isin(PAPER_IVS)].copy()
rq2["family"] = rq2["hypothesis"].str[:2]  # "H1" or "H2"

# canonical project order, with any unexpected names appended at the end
_present = list(rq2["project"].unique())
ORDERED = [p for p in PROJECT_ORDER if p in _present] + \
          [p for p in _present if p not in PROJECT_ORDER]

FAMILIES = [("H1", "H1a and H1b: days to acceptance"),
            ("H2", "H2a and H2b: acceptance likelihood")]
IV_STYLE = {"Mean CC": dict(marker="o", color="#1f77b4"),
            "Total SLOC": dict(marker="s", color="#d62728")}

print(f"{len(rq2)} (project, hypothesis, IV) rows across {len(ORDERED)} projects")

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
y = np.arange(len(ORDERED))
H = 0.38

for ax, (fam, title) in zip(axes, FAMILIES):
    sub = rq2[rq2["family"] == fam]
    tau_by = {(r["project"], r["iv"]): (r["tau"], r["significant"])
              for _, r in sub.iterrows()}
    for k, iv in enumerate(PAPER_IVS):
        st = IV_STYLE[iv]
        off = (H / 2) * (1 if k == 0 else -1)
        taus = [tau_by.get((p, iv), (np.nan, False))[0] for p in ORDERED]
        sigs = [tau_by.get((p, iv), (np.nan, False))[1] for p in ORDERED]
        faces = [st["color"] if s else "white" for s in sigs]
        ax.barh(y - off, taus, height=H, color=faces, label=iv,
                edgecolor=st["color"], linewidth=1.3)
    ax.axvline(0, color="gray", lw=1)
    ax.set_xlim(-0.55, 0.55)
    ax.set_xlabel(r"Kendall's $\tau$")
    ax.set_title(title)

axes[0].set_yticks(y)
axes[0].set_yticklabels(ORDERED)
axes[0].invert_yaxis()

from matplotlib.patches import Patch

handles = [Patch(facecolor="#1f77b4", edgecolor="#1f77b4", label="Mean CC"),
           Patch(facecolor="#d62728", edgecolor="#d62728", label="Total SLOC"),
           Patch(facecolor="white", edgecolor="gray", label="not significant")]
axes[1].legend(handles=handles, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(PLOTS / "rq2-bars.svg")
plt.show()

In [ ]:
complexity_conn.close()
proposals_conn.close()